# Run: tiny_cnn + MNIST, 20 rounds, checkpointed

One notebook, run top to bottom, single Colab session (free tier).
This is the ONLY notebook you need for this run -- it's a consolidated,
linear version of `00_Setup_and_Crypto_Tests.ipynb` + `CASE1_IID_MNIST.ipynb`
with the smoke test and the 2nd backbone removed, so there's nothing to
skip or get out of order.

**Scope** (see `FIX.md`/`STATUS.md`, 2026-11-15 entries, for the full
reasoning): 1 model -- `tiny_cnn` (~20.6K params), NOT resnet18 --
matching the model scale used by this paper's actual peer group
(WhiteCloak NDSS'26, PQS-BFL 2025, both ~19-22K params on MNIST), on
MNIST, `n_clients=50` (all 50 go through the crypto pipeline every
round -- `sampling_rate` only gates real-vs-zero local training, not
crypto participation), 20 rounds. Real measured per-round cost with
resnet18 was ~72 min/round (~24h for 20 rounds) because resnet18 is
~500x bigger than every comparable paper's model; tiny_cnn should cut
that dramatically since mask/sketch/range-proof cost is O(model_dim).

**If Colab disconnects partway through:** re-run from cell 1 again.
Drive stays mounted and the repo stays cloned across a disconnect/
reconnect, and the LAST cell detects the checkpoint already on Drive
and resumes after the last completed round -- it does not restart
from round 0. If you get a `NameError` for a variable partway through
re-running, it just means you haven't reached that cell yet this
pass -- keep running forward, don't skip around.

In [ ]:
# 1. Mount Drive (checkpoint durability + MNIST cache) and clone/pull the repo.
from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP

In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).

In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())

In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)

In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.mnist.source is "csv" (reads a
# Drive-cached CSV that this run's Drive doesn't have) -- that path is
# for whoever already staged the CSV. This run instead forces
# source="torchvision", which downloads MNIST directly over the network,
# no Drive dependency at all. Deliberately NOT changed in the shared
# config file, since other runs may rely on the csv path being default.
from fl.data_iid import prepare_case1

case1_data = prepare_case1(seed=CFG['seed'], source="torchvision")
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")

In [ ]:
# 7. Build the model.
# tiny_cnn (~20.6K params), not resnet18: matches the model scale used
# by this paper's actual peer group (WhiteCloak NDSS'26: 19K-param CNN
# on MNIST; PQS-BFL 2025: ~21.8K-param CNN on MNIST) -- see
# fl/models.py::build_tiny_cnn's docstring for the full citation
# reasoning. ~500x fewer params than resnet18's 11.2M, which is most of
# why this run is dramatically cheaper per round.
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)
flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


In [ ]:
# 8. The real run: 20 rounds, checkpointed every round to Drive.
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint

REAL_ROUNDS = 20  # down from CFG['mvp_scope']['rounds_per_run'] (45) -- see markdown cell 0

CKPT_DIR = '/content/drive/MyDrive/FL-ZKP/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = f'{CKPT_DIR}/case1_{REAL_MODEL}_r{REAL_ROUNDS}.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"bootstrap-{REAL_MODEL}".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
)
release_gpu_memory()
n_accepted = sum(1 for h in real_history if not h['aborted'])
print(f"done: {len(real_history)} rounds in history, {n_accepted} accepted (not aborted)")

## Evaluate accuracy

Works whether you let training finish, stopped it early (Ctrl-C /
interrupt), or are coming back in a fresh runtime with nothing but the
checkpoint on Drive: it loads the LATEST checkpointed weights directly
from `CKPT_PATH` (falling back to the in-memory `final_w_t` only if no
checkpoint file exists yet), so it always reports accuracy for
whatever round you actually reached, not a partially-updated `model`
object. Reusable as-is for `small_resnet` later -- every name it reads
(`REAL_MODEL`, `model`, `model_shapes`, `CKPT_PATH`, `case1_data`) is
already generic, so nothing here needs to change when you switch
backbones.


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            logits = model(images[i:i + batch_size])
            preds = logits.argmax(dim=1)
            correct += (preds == labels[i:i + batch_size]).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- this is what makes
# the cell correct even if you stopped training early: `final_w_t`/
# `model` in memory only reflect whatever round your training cell
# actually got through before you interrupted it, but so does the
# checkpoint (it's written after every round), so reading from
# CKPT_PATH and reading from memory agree whenever both exist. Preferring
# the checkpoint means this cell also works standalone after a fresh
# runtime restart, with no retraining needed.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t, model_shapes))

# Official MNIST test set (10,000 images) -- the headline number.
test_images, test_labels = case1_data['test']
test_acc = evaluate_accuracy(model, test_images, test_labels)

# global_val: a held-out slice of the TRAINING set (disjoint from every
# client's own data, see fl/splits.py) -- a secondary sanity check, not
# the headline number, since it's drawn from the same distribution the
# clients trained on rather than the official test split.
val_ds = case1_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: official MNIST test accuracy = {test_acc:.4f} ({len(test_images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")
